# 🚀 Çok Ajanlı Fikir Test Etme Sistemi (Jupyter Notebook Edition)
Bu notebook, bir ürün veya girişim fikrini **5 otonom yapay zekâ ajanı** ile canlı araştırmalar yaparak **GO / CONDITIONAL GO / NO-GO** raporuna dönüştürür.

In [ ]:
# app/server.py - RAG Destekli Hakeme Soru Sor API'si & Stripe Ödeme Webhook
@app.post("/api/ask-judge")
def ask_judge_followup(req: AskJudgeRequest):
    job = JOBS.get(req.job_id)
    if not job or job["status"] != "completed":
        raise HTTPException(status_code=400, detail="Tamamlanmış analiz bulunamadı.")
    
    # RAG Motorundan Bağlam Parçalarını Çek (Grounding)
    rag_context = rag_engine.retrieve(req.question, top_k=2)
    prompt = f"RAG Grounding Context:\n{rag_context}\n\nFounder Question: {req.question}"
    
    judge_agent = get_judge_agent()
    qa_task = Task(description=prompt, expected_output="RAG backed advice.", agent=judge_agent)
    qa_crew = Crew(agents=[judge_agent], tasks=[qa_task], verbose=False)
    
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    answer = loop.run_until_complete(qa_crew.kickoff_async())
    loop.close()
    return {"answer": str(answer)}

## 📦 1. Adım: Paket Yükleme

In [6]:
# Bağımlılıkları yükleyin (İlk çalıştırmada gereklidir)
!pip install crewai crewai-tools langchain-google-genai duckduckgo-search python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


## 🔑 2. Adım: Gemini API Anahtarı ve LLM Ayarları

In [7]:
import os
from crewai import LLM

# ⚠️ Buraya kendi Google Gemini API Key'inizi yazın (https://aistudio.google.com/)
os.environ["OPEN_AI_API_KEY"] = "your-openai-api-key"

# LLM Tanımlama (Gemini 1.5 Flash)
llm = LLM(
    model="openai/gpt-4o-mini",
    api_key=os.environ.get("OPEN_AI_API_KEY"),
    temperature=0.7
)
print("✅ LLM başarıyla yapılandırıldı!")

✅ LLM başarıyla yapılandırıldı!


In [8]:
%pip install duckduckgo-search crewai crewai-tools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


## 🛠️ 3. Adım: Özel Ajan Araçlarının (Custom Tools) Tanımlanması

In [10]:
from crewai.tools import tool
from duckduckgo_search import DDGS

@tool("Web Search Tool")
def web_search_tool(query: str) -> str:
    """Searches the live web using DuckDuckGo for competitor info, market trends, user reviews, or news."""
    try:
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                results.append(f"Title: {r.get('title')}\nURL: {r.get('href')}\nSnippet: {r.get('body')}\n")
        return "\n---\n".join(results) if results else f"No search results found for: '{query}'"
    except Exception as e:
        return f"Error executing search: {str(e)}"

@tool("GitHub Search Tool")
def github_search_tool(query: str) -> str:
    """Searches GitHub repositories for open source libraries, tools, and technical precedents."""
    try:
        search_query = f"{query} site:github.com"
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(search_query, max_results=4):
                results.append(f"Repo/Project: {r.get('title')}\nURL: {r.get('href')}\nSummary: {r.get('body')}\n")
        return "\n---\n".join(results) if results else f"No GitHub repos found for: '{query}'"
    except Exception as e:
        return f"Error searching GitHub: {str(e)}"

@tool("Infrastructure & API Cost Calculator")
def cost_calculator_tool(user_scale_tier: str) -> str:
    """Calculates estimated monthly API and cloud infrastructure operational costs."""
    tier = user_scale_tier.lower().strip()
    if "growth" in tier or "10k" in tier:
        return "### Monthly Cost (Growth Tier - 10k users):\n- Hosting: $50-$120/mo\n- API Tokens: $150-$400/mo\n- Total: $245-$610/mo"
    elif "scale" in tier or "100k" in tier:
        return "### Monthly Cost (Scale Tier - 100k users):\n- Hosting: $450-$1,200/mo\n- API Tokens: $1,500-$4,500/mo\n- Total: $2,250-$6,500/mo"
    else:
        return "### Monthly Cost (MVP Tier - 1k users):\n- Hosting: $0-$15/mo\n- API Tokens: $5-$35/mo\n- Total: $5-$50/mo"

print("✅ Özel araçlar (Web Search, GitHub Search, Cost Calculator) tanımlandı!")

✅ Özel araçlar (Web Search, GitHub Search, Cost Calculator) tanımlandı!


In [3]:
#şimdi test edelim toollar çalışıyor mu ? 
web_search_tool.run("Shopify AI return apps")

C:\Users\SPECIAL TEK\AppData\Local\Temp\ipykernel_3856\4114778027.py:9: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


'Title: Shopify: The All-in-One Commerce Platform for Businesses - Shopify\nURL: https://www.shopify.com/\nSnippet: Try Shopify free. Build or grow your business fast with AI. Get more than ecommerce software with tools to manage every part of your business.\n\n---\nTitle: Build Your Online Store: Use Themes or Go Headless - Shopify\nURL: https://www.shopify.com/online\nSnippet: Whether you’re looking to build a store from scratch or want something more move-in ready, Shopify has options for every business. Both now and as you grow.\n\n---\nTitle: Shopify - Wikipedia\nURL: https://en.wikipedia.org/wiki/Shopify\nSnippet: On August 14, 2019, Shopify launched Shopify Chat, a new native chat function that allows merchants to have real-time conversations with customers visiting Shopify stores online.\n\n---\nTitle: Shopify – Wikipedia\nURL: https://de.wikipedia.org/wiki/Shopify\nSnippet: Shopify ist eine proprietäre E-Commerce -Software, die von dem gleichnamigen kanadischen Unternehmen ver

In [4]:
github_search_tool.run("ecommerce return bot")

C:\Users\SPECIAL TEK\AppData\Local\Temp\ipykernel_3856\4114778027.py:22: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


"No GitHub repos found for: 'ecommerce return bot'"

## 🤖 4. Adım: 5 Otonom Ajanın Oluşturulması

In [11]:
from crewai import Agent

# 1. Market & Competitor Agent
market_agent = Agent(
    role="Senior Market Analyst & Competitive Intelligence Lead",
    goal="Identify top competitors, market gaps, market size, and Unique Value Proposition (USP).",
    backstory="Experienced VC market analyst specializing in competitive intelligence and market dynamics.",
    tools=[web_search_tool],
    llm=llm,
    verbose=True
)

# 2. Demand & Sentiment Agent
demand_agent = Agent(
    role="Consumer Demand & Sentiment Specialist",
    goal="Uncover real user pain points, community sentiment, and existing demand patterns.",
    backstory="Digital customer researcher expert in online discussions, customer reviews, and search intent.",
    tools=[web_search_tool],
    llm=llm,
    verbose=True
)

# 3. Technical Feasibility Agent
tech_agent = Agent(
    role="Principal Software Architect & Technical Feasibility Lead",
    goal="Assess technical complexity, recommend tech stack, leverage open-source code, and estimate MVP build time.",
    backstory="Veteran CTO with expertise in full-stack architecture, AI system integration, and rapid prototyping.",
    tools=[github_search_tool, web_search_tool],
    llm=llm,
    verbose=True
)

# 4. Financial & Legal Agent
financial_agent = Agent(
    role="Fintech Cost Modeler & Compliance Auditor",
    goal="Estimate infrastructure/API costs, model unit economics, and highlight regulatory risks (KVKK, GDPR).",
    backstory="Technology CFO and legal compliance strategist analyzing business model sustainability.",
    tools=[cost_calculator_tool],
    llm=llm,
    verbose=True
)

# 5. Judge Agent (Decision Engine)
judge_agent = Agent(
    role="Chief Investment Officer & System Judge",
    goal="Synthesize reports from all research agents, apply weighted scoring, and render GO / CONDITIONAL GO / NO-GO verdict.",
    backstory="Top-tier investment committee judge known for data-driven and pragmatic evaluations.",
    tools=[],
    llm=llm,
    verbose=True
)

print("✅ 5 Otonom Ajan başarıyla oluşturuldu!")

✅ 5 Otonom Ajan başarıyla oluşturuldu!


c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\crewai\agent\core.py:367: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\crewai\agent\core.py:375: DeprecationWarning: deprecated
  if self.allow_code_execution:
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\crewai\agent\core.py:386: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\crewai\agent\core.py:367: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\crewai\agent\core.py:375: DeprecationWarning: deprecated
  

## 📝 5. Adım: Test Edilecek Fikrin Belirlenmesi

In [12]:
# Buradaki bilgileri kendi fikirlerinize göre değiştirebilirsiniz:
IDEA_TITLE = "AI E-Ticaret İade Asistanı"
IDEA_DESCRIPTION = "E-ticaret siteleri için müşteri iadelerini otomatik yöneten, kargo entegrasyonu sağlayan ve dolandırıcılık tespiti yapan otonom yapay zeka asistanı."
TARGET_AUDIENCE = "Shopify ve WooCommerce kullanan orta ölçekli e-ticaret mağaza sahipleri"

print(f"📌 Test Fikri: {IDEA_TITLE}")
print(f"🎯 Hedef Kitle: {TARGET_AUDIENCE}")

📌 Test Fikri: AI E-Ticaret İade Asistanı
🎯 Hedef Kitle: Shopify ve WooCommerce kullanan orta ölçekli e-ticaret mağaza sahipleri


## 📋 6. Adım: Görevlerin (Tasks) Tanımlanması

In [13]:
from crewai import Task

task_market = Task(
    description=f"Conduct market research for '{IDEA_TITLE}' ({IDEA_DESCRIPTION}). Find at least 3 competitors, market gaps, and USP.",
    expected_output="Competitor Table, Market Gap Analysis, and USP Recommendation.",
    agent=market_agent
)

task_demand = Task(
    description=f"Analyze user demand and pain points for '{IDEA_TITLE}' targeting '{TARGET_AUDIENCE}'. Rate demand urgency (1-10).",
    expected_output="Top 3 Pain Points, Community Sentiment, and Urgency Score.",
    agent=demand_agent
)

task_tech = Task(
    description=f"Evaluate technical feasibility for '{IDEA_TITLE}'. Recommend tech stack, GitHub repos, and estimate MVP build duration.",
    expected_output="Recommended Tech Stack, Open-Source Repos, and MVP Timeline.",
    agent=tech_agent
)

task_financial = Task(
    description=f"Estimate monthly API/cloud costs (MVP vs Growth vs Scale) and compliance risks (GDPR/KVKK) for '{IDEA_TITLE}'.",
    expected_output="Monthly Cost Estimates, Monetization Model, and Legal Risks.",
    agent=financial_agent
)

task_judge = Task(
    description=(
        f"Synthesize reports from all research agents for '{IDEA_TITLE}'.\n"
        "1. Calculate an Overall Idea Score (0-100).\n"
        "2. Render a verdict: **GO** (Score >= 75), **CONDITIONAL GO** (Score 55-74), or **NO-GO** (Score < 55).\n"
        "3. Provide top 3 action items or pivot advice."
    ),
    expected_output="Comprehensive Executive Verdict Report with Final Score, GO/NO-GO Verdict, and Action Plan.",
    agent=judge_agent
)

print("✅ Tüm görevler başarıyla bağlandı!")

✅ Tüm görevler başarıyla bağlandı!


## 🚀 7. Adım: Crew'i Çalıştırma & Hakem Raporu Alma

In [15]:
from crewai import Crew, Process
from IPython.display import Markdown

crew = Crew(
    agents=[market_agent, demand_agent, tech_agent, financial_agent, judge_agent],
    tasks=[task_market, task_demand, task_tech, task_financial, task_judge],
    process=Process.sequential,
    verbose=True
)

print("🚀 Ajanlar çalışmaya başlıyor... (Bu işlem 2-4 dakika sürebilir)")

# ⚠️ Jupyter için `await` ve `kickoff_async()` kullanıyoruz:
result = await crew.kickoff_async()

# Sonucu ekranda formatlı Markdown olarak göster
Markdown(str(result))

c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\crewai\crew.py:643: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):


🚀 Ajanlar çalışmaya başlıyor... (Bu işlem 2-4 dakika sürebilir)


c:\Users\SPECIAL TEK\AppData\Local\Programs\Python\Python313\Lib\json\decoder.py:361: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000002CFF2552980>
  obj, end = self.scan_once(s, idx)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.12                                                                                       │
│  Latest version:  1.15.14                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9abe5d85-cc42-4373-8d05-4bbb4e1712a9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer & System Judge                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Comprehensive Executive Verdict Report for 'AI E-Ticaret İade Asistanı'                                    │
│                                                                                                                 │
│  #### Overall Idea Score Calculation                                                                            │
│                                                                                                                 │
│  To calculate the Overall Idea Score, I will assess various components based on criteria including market       │
│  potential, uniqueness, technical feasibility, community sentiment, and urgency.                                │
│                                                                                                                 │
│  1. **Market Potential (25 points)**                                                                            │
│     - There is a clear need for efficient return management solutions in e-commerce due to increasing returns   │
│  and fraud concerns.                                                                                            │
│     - Score: **22/25**                                                                                          │
│                                                                                                                 │
│  2. **Uniqueness of Value Proposition (20 points)**                                                             │
│     - The proposed solution addresses critical gaps such as customization, fraud detection, and multi-channel   │
│  support.                                                                                                       │
│     - Score: **18/20**                                                                                          │
│                                                                                                                 │
│  3. **Technical Feasibility (20 points)**                                                                       │
│     - The recommended tech stack and MVP timeline are robust and achievable within the proposed timeframe.      │
│     - Score: **16/20**                                                                                          │
│                                                                                                                 │
│  4. **Community Sentiment (15 points)**                                                                         │
│     - The sentiment is mixed, with enthusiasm for AI but concerns over complexity and support.                  │
│     - Score: **10/15**                                                                                          │
│                                                                                                                 │
│  5. **Urgency Score (20 points)**                                                                               │
│     - High urgency due to the current market dynamics, indicating a strong demand for such a solution.          │
│     - Score: **18/20**                                                                                          │
│                                                                                                                 │
│  **Total Score Calculation:**                          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Synthesize reports from all research agents for 'AI E-Ticaret İade Asistanı'.                            │
│  1. Calculate an Overall Idea Score (0-100).                                                                    │
│  2. Render a verdict: **GO** (Score >= 75), **CONDITIONAL GO** (Score 55-74), or **NO-GO** (Score < 55).        │
│  3. Provide top 3 action items or pivot advice.                                                                 │
│  Agent: Chief Investment Officer & System Judge                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9abe5d85-cc42-4373-8d05-4bbb4e1712a9                                                                       │
│  Final Output: ### Comprehensive Executive Verdict Report for 'AI E-Ticaret İade Asistanı'                      │
│                                                                                                                 │
│  #### Overall Idea Score Calculation                                                                            │
│                                                                                                                 │
│  To calculate the Overall Idea Score, I will assess various components based on criteria including market       │
│  potential, uniqueness, technical feasibility, community sentiment, and urgency.                                │
│                                                                                                                 │
│  1. **Market Potential (25 points)**                                                                            │
│     - There is a clear need for efficient return management solutions in e-commerce due to increasing returns   │
│  and fraud concerns.                                                                                            │
│     - Score: **22/25**                                                                                          │
│                                                                                                                 │
│  2. **Uniqueness of Value Proposition (20 points)**                                                             │
│     - The proposed solution addresses critical gaps such as customization, fraud detection, and multi-channel   │
│  support.                                                                                                       │
│     - Score: **18/20**                                                                                          │
│                                                                                                                 │
│  3. **Technical Feasibility (20 points)**                                                                       │
│     - The recommended tech stack and MVP timeline are robust and achievable within the proposed timeframe.      │
│     - Score: **16/20**                                                                                          │
│                                                                                                                 │
│  4. **Community Sentiment (15 points)**                                                                         │
│     - The sentiment is mixed, with enthusiasm for AI but concerns over complexity and support.                  │
│     - Score: **10/15**                                                                                          │
│                                                                                                                 │
│  5. **Urgency Score (20 points)**                                                                               │
│     - High urgency due to the current market dynamics, indicating a strong demand for such a solution.          │
│     - Score: **18/20**                                                                                          │
│                                                                                                                 │
│  **Total Score Calculation:**                         

### Comprehensive Executive Verdict Report for 'AI E-Ticaret İade Asistanı'

#### Overall Idea Score Calculation

To calculate the Overall Idea Score, I will assess various components based on criteria including market potential, uniqueness, technical feasibility, community sentiment, and urgency.

1. **Market Potential (25 points)**
   - There is a clear need for efficient return management solutions in e-commerce due to increasing returns and fraud concerns.
   - Score: **22/25**

2. **Uniqueness of Value Proposition (20 points)**
   - The proposed solution addresses critical gaps such as customization, fraud detection, and multi-channel support.
   - Score: **18/20**

3. **Technical Feasibility (20 points)**
   - The recommended tech stack and MVP timeline are robust and achievable within the proposed timeframe.
   - Score: **16/20**

4. **Community Sentiment (15 points)**
   - The sentiment is mixed, with enthusiasm for AI but concerns over complexity and support.
   - Score: **10/15**

5. **Urgency Score (20 points)**
   - High urgency due to the current market dynamics, indicating a strong demand for such a solution.
   - Score: **18/20**

**Total Score Calculation:**
- Market Potential: 22
- Uniqueness: 18
- Technical Feasibility: 16
- Community Sentiment: 10
- Urgency: 18

**Overall Idea Score = 22 + 18 + 16 + 10 + 18 = 84/100**

#### Verdict
**GO** (Score: 84)

#### Top 3 Action Items or Pivot Advice

1. **Focus on Integration Capabilities:**
   - Prioritize the development of seamless integration with popular e-commerce platforms and shipping providers to resolve integration challenges highlighted in the pain points. This will enhance user adoption and satisfaction.

2. **Develop Advanced Fraud Detection Features:**
   - Invest in the development of sophisticated AI algorithms for fraud detection that go beyond basic measures. This will meet market demand and set the product apart from existing competitors.

3. **Create Customizable User Interfaces:**
   - Ensure that the platform allows for extensive customization options that enable e-commerce businesses to align the return process with their branding and customer service standards. This will enhance the overall customer experience.

### Conclusion
The 'AI E-Ticaret İade Asistanı' presents a compelling opportunity to address critical needs in e-commerce return management. With a solid plan for development and a clear understanding of market dynamics, the initiative is well-positioned for success. Prioritizing integration, advanced fraud detection, and customization will further enhance its value proposition in the competitive landscape.

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯